In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as geom_nn
from torch_geometric.data import Data, Dataset
import h5py
import numpy as np
import itertools

# ==========================================
# 1. Hardware & GPU Acceleration
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

# ==========================================
# 2. Programmatic Kagome Graph Generator
# ==========================================
class KagomeGraphGenerator:
    def __init__(self, Lx, Ly, bc_x, bc_y, t1, t2):
        self.Lx = Lx
        self.Ly = Ly
        self.bc_x = bc_x
        self.bc_y = bc_y
        self.t1 = t1
        self.t2 = t2
        self.num_sites = Lx * Ly * 3
        
        # Kagome basis vectors and site positions within unit cell
        self.a1 = np.array([1.0, 0.0])
        self.a2 = np.array([0.5, np.sqrt(3)/2])
        self.basis = np.array([
            [0.0, 0.0],                  # Site 0 (A)
            [0.5, 0.0],                  # Site 1 (B)
            [0.25, np.sqrt(3)/4]         # Site 2 (C)
        ])

    def generate_coordinates(self):
        coords = []
        for y in range(self.Ly):
            for x in range(self.Lx):
                R = x * self.a1 + y * self.a2
                for i in range(3):
                    coords.append(R + self.basis[i])
        return torch.tensor(np.array(coords), dtype=torch.float32, device=device)

    def generate_edges_and_weights(self):
        # Structurally maps t1 (NN) and t2 (NNN) hopping terms.
        # Boundary edges are flagged for TBC wrapping later.
        edge_indices = []
        edge_weights = []
        is_boundary = []
        
        # MOCK IMPLEMENTATION of lattice connectivity for modularity
        # A full Kagome neighbor search maps intra-cell and inter-cell sites.
        # Here we programmatically structure the arrays.
        for i in range(self.num_sites):
            for j in range(i + 1, self.num_sites):
                # Placeholder logic: distance-based NN/NNN identification
                # In production, exact indexing based on (x, y, basis) is used.
                dist = 1.0 # Mock distance
                if dist == 0.5: # t1 (nearest neighbor)
                    edge_indices.extend([[i, j], [j, i]])
                    edge_weights.extend([self.t1, self.t1])
                    is_boundary.extend([0, 0]) 
                elif dist > 0.5 and dist < 1.0: # t2 (next-nearest)
                    edge_indices.extend([[i, j], [j, i]])
                    edge_weights.extend([self.t2, self.t2])
                    is_boundary.extend([0, 0])
        
        edge_index = torch.tensor(edge_indices, dtype=torch.long, device=device).t()
        edge_attr = torch.tensor(edge_weights, dtype=torch.float32, device=device)
        is_boundary = torch.tensor(is_boundary, dtype=torch.bool, device=device)
        return edge_index, edge_attr, is_boundary

# ==========================================
# 3. MoE Data Pipeline & TBC Wrapping
# ==========================================
class FermiHubbardDataset(Dataset):
    def __init__(self, h5_path):
        super().__init__(root=None, transform=None, pre_transform=None)
        self.h5_path = h5_path
        # Example loading logic from h5
        # self.data_file = h5py.File(self.h5_path, 'r')
        # self.keys = list(self.data_file.keys())
        self.keys = [] # Placeholder for actual dataset length

    def len(self):
        return len(self.keys)

    def get(self, idx):
        # 11 input params: Lx, Ly, bc_x, bc_y, t1, t2, U, cons_N, cons_Sz, theta_x, theta_y
        # MOCK DATA LOADING
        params = torch.rand(11, dtype=torch.float32) 
        Lx, Ly, bc_x, bc_y, t1, t2 = int(params[0]), int(params[1]), params[2], params[3], params[4], params[5]
        theta_x, theta_y = params[9], params[10]
        
        generator = KagomeGraphGenerator(Lx, Ly, bc_x, bc_y, t1, t2)
        coords = generator.generate_coordinates()
        edge_index, base_edge_weights, is_boundary = generator.generate_edges_and_weights()
        
        # Apply Twisted Boundary Conditions (TBC) to boundary edges
        complex_weights = torch.complex(base_edge_weights, torch.zeros_like(base_edge_weights))
        phase_x = torch.exp(1j * theta_x)
        phase_y = torch.exp(1j * theta_y)
        
        # In actual execution, is_boundary would differentiate between x and y wrapping
        complex_weights[is_boundary] *= phase_x # Simplified mapping
        
        node_features = torch.ones((generator.num_sites, 1), dtype=torch.cfloat, device=device) * params[6] # Include U
        
        data = Data(x=node_features, edge_index=edge_index, edge_attr=complex_weights, pos=coords)
        data.params = params.to(device)
        
        # Targets: Energy, D_occ, n_i, SpinCorr, ChargeCorr
        # Mocking true labels
        data.y_E = torch.tensor(0.0)
        data.y_Docc = torch.zeros(generator.num_sites)
        data.y_n = torch.zeros(generator.num_sites)
        data.y_SCorr = torch.zeros((generator.num_sites, generator.num_sites))
        data.y_CCorr = torch.zeros((generator.num_sites, generator.num_sites))
        
        return data

# ==========================================
# 4. Expert Router (MoE Gating)
# ==========================================
class MoERouter(nn.Module):
    def __init__(self, input_dim=11, num_experts=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, num_experts)
        )
    def forward(self, x):
        # Outputs a softmax distribution over experts: Weak, Transition, Strong
        return F.softmax(self.net(x), dim=-1)

# ==========================================
# 5. Size-Invariant GNN Experts
# ==========================================
class ComplexGNNExpert(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        # PyG standard layers don't natively handle cfloat well out of the box, 
        # so we split real and imaginary parts for message passing.
        self.conv1 = geom_nn.GCNConv(2, hidden_dim) # node feats (real, imag)
        self.conv2 = geom_nn.GCNConv(hidden_dim, hidden_dim)
        
    def forward(self, x, edge_index, edge_attr):
        # x: (N, 1) cfloat -> (N, 2) float
        x_real_imag = torch.cat([x.real, x.imag], dim=-1)
        
        # Edge weights (cfloat) -> absolute magnitude for standard conv
        # Or you can map real/imag edge features for custom MessagePassing.
        edge_weight_mag = torch.abs(edge_attr)
        
        h = F.relu(self.conv1(x_real_imag, edge_index, edge_weight_mag))
        h = self.conv2(h, edge_index, edge_weight_mag)
        return h # Node embeddings (N, hidden_dim)

# ==========================================
# 6. Output Decoding & Blending
# ==========================================
class KagomeMoEGNN(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.router = MoERouter(input_dim=11, num_experts=3)
        self.experts = nn.ModuleList([ComplexGNNExpert(hidden_dim) for _ in range(3)])
        
        # Decoders (Size-invariant via global pooling or node mapping)
        self.energy_decoder = nn.Sequential(nn.Linear(hidden_dim, 16), nn.ReLU(), nn.Linear(16, 1))
        self.docc_decoder = nn.Linear(hidden_dim, 1)
        self.density_decoder = nn.Linear(hidden_dim, 1)
        
        # Correlation Matrices Decoder (Bilinear/MLP over node pairs)
        self.corr_proj = nn.Linear(hidden_dim, int(hidden_dim/2))

    def forward(self, data):
        # 1. Route
        routing_weights = self.router(data.params.unsqueeze(0)).squeeze(0) # [w1, w2, w3]
        
        # 2. Expert Forward Passes
        expert_outputs = []
        for expert in self.experts:
            expert_outputs.append(expert(data.x, data.edge_index, data.edge_attr))
            
        # 3. Blend node embeddings
        blended_nodes = torch.zeros_like(expert_outputs[0])
        for w, out in zip(routing_weights, expert_outputs):
            blended_nodes += w * out
            
        # 4. Decode Observables
        # Energy (Global via mean pooling) -> Scalar
        global_embed = torch.mean(blended_nodes, dim=0, keepdim=True)
        E_pred = self.energy_decoder(global_embed).squeeze()
        
        # Local observables -> Node Vectors
        Docc_pred = self.docc_decoder(blended_nodes).squeeze()
        n_pred = self.density_decoder(blended_nodes).squeeze()
        
        # Two-point correlations -> Matrices (N x N)
        corr_embeds = self.corr_proj(blended_nodes)
        # S_ij = h_i * h_j^T (Size invariant outer product projection)
        SCorr_pred = torch.matmul(corr_embeds, corr_embeds.t())
        CCorr_pred = torch.matmul(corr_embeds, corr_embeds.t()) 
        
        return E_pred, Docc_pred, n_pred, SCorr_pred, CCorr_pred

# ==========================================
# 7. Analytic Structure Factors
# ==========================================
def compute_structure_factor(corr_matrix, coords, q):
    """
    Analytically computes S(q) via Fourier transform.
    S(q) = (1/N) * Sum_{i,j} e^{i q . (r_i - r_j)} * C_{ij}
    """
    N = coords.shape[0]
    # (N, N, 2) relative coordinate vectors
    diffs = coords.unsqueeze(1) - coords.unsqueeze(0) 
    
    # Dot product with q: q_x * diff_x + q_y * diff_y
    dot_products = diffs[:, :, 0] * q[0] + diffs[:, :, 1] * q[1]
    
    # Exponential phase factor
    phases = torch.exp(1j * dot_products)
    
    # Compute structure factor at q
    S_q = (1.0 / N) * torch.sum(phases * corr_matrix)
    return torch.abs(S_q) # Return real observable

# ==========================================
# 8. TBC Averaging Loop
# ==========================================
def tbc_averaging_forward(model, data):
    tbc_phases = [(0.0, 0.0), (np.pi, 0.0), (0.0, np.pi), (np.pi, np.pi)]
    
    avg_E, avg_Docc, avg_n, avg_SCorr, avg_CCorr = 0, 0, 0, 0, 0
    
    for (theta_x, theta_y) in tbc_phases:
        # Clone data and alter TBC input params temporarily
        mod_data = data.clone()
        mod_data.params[9] = theta_x
        mod_data.params[10] = theta_y
        
        # Adjust edge weights for TBC (Simplified wrap)
        phase_x = torch.exp(torch.tensor(1j * theta_x, device=device))
        mod_data.edge_attr[mod_data.is_boundary] = data.edge_attr[mod_data.is_boundary] * phase_x 
        
        E, Docc, n, SCorr, CCorr = model(mod_data)
        
        avg_E += E / 4.0
        avg_Docc += Docc / 4.0
        avg_n += n / 4.0
        avg_SCorr += SCorr / 4.0
        avg_CCorr += CCorr / 4.0
        
    return avg_E, avg_Docc, avg_n, avg_SCorr, avg_CCorr

# ==========================================
# 9. GATE G (Physics Verification Module)
# ==========================================
def gate_g_physics_verification(n_pred, CCorr_pred, SCorr_pred, coords, cons_N, cons_Sz):
    violation_flag = False
    
    # 1. Conservation Error
    total_particles = torch.sum(n_pred)
    if not torch.isclose(total_particles, cons_N, atol=1e-1):
        violation_flag = True
        
    # Simplified Sz check (assuming n_pred relates to spin imbalance if projected)
    # If cons_Sz == 0, we expect certain symmetries in the spin structure.
    
    # 2. Disconnected-Part Error (Connected Correlation Decay)
    # C_connected_ij = C_ij - <n_i><n_j>
    connected_corr = CCorr_pred - torch.outer(n_pred, n_pred)
    # Physical requirement: sum of connected correlations should relate to charge variance
    # Flag if unphysical explosive correlations occur
    if torch.max(torch.abs(connected_corr)) > 10.0: 
        violation_flag = True

    # 3. Symmetry Artifact
    # Check for square-lattice artifact at exactly (pi, pi)
    q_artifact = torch.tensor([np.pi, np.pi], device=device)
    Sq_artifact = compute_structure_factor(SCorr_pred, coords, q_artifact)
    
    # Compare with another arbitrary q to ensure it's not falsely acting as a global peak
    q_baseline = torch.tensor([0.0, 0.0], device=device)
    Sq_baseline = compute_structure_factor(SCorr_pred, coords, q_baseline)
    
    if Sq_artifact > (Sq_baseline * 1.5): # False global peak threshold
        violation_flag = True
        
    return violation_flag

# ==========================================
# 10. Heavily Penalized Loss Function
# ==========================================
def compute_penalized_loss(preds, targets, data, coords, cons_N, cons_Sz):
    E_pred, Docc_pred, n_pred, SCorr_pred, CCorr_pred = preds
    E_t, Docc_t, n_t, SCorr_t, CCorr_t = targets
    
    # Base MSE Loss
    loss_E = F.mse_loss(E_pred, E_t)
    loss_Docc = F.mse_loss(Docc_pred, Docc_t)
    loss_n = F.mse_loss(n_pred, n_t)
    loss_SCorr = F.mse_loss(SCorr_pred, SCorr_t)
    loss_CCorr = F.mse_loss(CCorr_pred, CCorr_t)
    
    mse_total = loss_E + loss_Docc + loss_n + loss_SCorr + loss_CCorr
    
    # GATE G verification
    violation = gate_g_physics_verification(n_pred, CCorr_pred, SCorr_pred, coords, cons_N, cons_Sz)
    
    penalty_multiplier = 100.0 if violation else 1.0
    
    final_loss = mse_total * penalty_multiplier
    return final_loss

# ==========================================
# Execution Example / Main Routine
# ==========================================
if __name__ == "__main__":
    model = KagomeMoEGNN(hidden_dim=64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Mocking single dataset load
    dataset = FermiHubbardDataset("mock_path.h5")
    # Forcing a mock length of 1 for demonstration
    dataset.keys = ['mock_1'] 
    
    model.train()
    for step in range(5):
        optimizer.zero_grad()
        
        data = dataset[0]
        targets = (data.y_E, data.y_Docc, data.y_n, data.y_SCorr, data.y_CCorr)
        
        # Forward Pass with TBC Averaging
        preds = tbc_averaging_forward(model, data)
        
        # Calculate Penalized Loss
        cons_N, cons_Sz = data.params[7], data.params[8]
        loss = compute_penalized_loss(preds, targets, data, data.pos, cons_N, cons_Sz)
        
        loss.backward()
        optimizer.step()
        
        print(f"Step {step} | Loss: {loss.item():.4f}")

False

In [ ]:
# ==========================================
# Testing / Evaluation Phase
# ==========================================
model.eval() # 1. Set model to evaluation mode
total_test_mse = 0.0
physical_violations = 0

# 2. Disable gradient tracking
with torch.no_grad(): 
    for data in test_dataset: # Assuming a separate holdout dataset
        targets = (data.y_E, data.y_Docc, data.y_n, data.y_SCorr, data.y_CCorr)
        
        # Forward Pass with TBC Averaging (same as training)
        preds = tbc_averaging_forward(model, data)
        E_pred, Docc_pred, n_pred, SCorr_pred, CCorr_pred = preds
        E_t, Docc_t, n_t, SCorr_t, CCorr_t = targets
        
        # 3. Calculate unpenalized MSE for pure performance tracking
        mse = (F.mse_loss(E_pred, E_t) + F.mse_loss(Docc_pred, Docc_t) + 
               F.mse_loss(n_pred, n_t) + F.mse_loss(SCorr_pred, SCorr_t) + 
               F.mse_loss(CCorr_pred, CCorr_t))
        total_test_mse += mse.item()
        
        # 4. Use GATE G strictly for evaluation logging
        cons_N, cons_Sz = data.params[7], data.params[8]
        if gate_g_physics_verification(n_pred, CCorr_pred, SCorr_pred, data.pos, cons_N, cons_Sz):
            physical_violations += 1

print(f"Test MSE: {total_test_mse / len(test_dataset):.4f}")
print(f"Physical Violations: {physical_violations} out of {len(test_dataset)}")